# SignBridge AI - Results
Reads `reports/metrics.json` (real data) or `reports/demo/metrics.json` (SYNTHETIC demo data). Nothing here is typed in by hand.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
import pandas as pd, matplotlib.pyplot as plt, matplotlib.image as mpimg
real = ROOT / "reports" / "metrics.json"; demo = ROOT / "reports" / "demo" / "metrics.json"
mp = real if real.exists() else demo
SYNTH = mp == demo
M = json.load(open(mp)); RD = mp.parent
print("using", mp, "(SYNTHETIC DEMO DATA)" if SYNTH else "")
rows = []
for lang, e in M["languages"].items():
    for kind in ("static", "dynamic"):
        if kind in e:
            for name, m in e[kind]["models"].items():
                rows.append(dict(language=lang, type=kind, model=name.upper(), accuracy=m["accuracy"], macro_f1=m["macro_f1"],
                                 latency_ms=m["latency_ms_per_sample_model_only"], split=e[kind]["split_mode"], n_test=e[kind]["n_test"]))
df = pd.DataFrame(rows); df.round(4)

## LSTM vs GRU ablation (same data, same split, same hyper-parameters)

In [ ]:
abl = df[df.type == "dynamic"].pivot(index="language", columns="model", values=["accuracy", "macro_f1", "latency_ms"]).round(4)
abl

## Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for lang in M["languages"]:
    p = RD / f"train_dynamic_{lang}.json"
    if p.exists():
        h = json.load(open(p))["history"]
        for k in ("lstm", "gru"):
            axes[0].plot(h[k]["val_loss"], label=f"{lang} {k}")
            axes[1].plot(h[k]["val_accuracy"], label=f"{lang} {k}")
axes[0].set_title("validation loss"); axes[1].set_title("validation accuracy"); axes[0].legend(fontsize=7); plt.tight_layout(); plt.show()

## Confusion matrices

In [ ]:
pngs = sorted(RD.glob("confusion_*.png"))
for p in pngs[:4]:
    plt.figure(figsize=(6, 5)); plt.imshow(mpimg.imread(p)); plt.axis("off"); plt.title(p.name); plt.show()
print(len(pngs), "confusion matrices in", RD)

## Speed

In [ ]:
pd.DataFrame([{**dict(language=l, type=k), **v} for l, e in M["languages"].items() for k, v in e.get("pipeline_latency", {}).items()]).round(2)

In [ ]:
print("Hand detector latency:", M["hand_detector_latency"])
print("Environment:", M["environment"])